In [3]:
from ingest import load_faq_data
documents = load_faq_data()

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [5]:
documents_llm[5]

{'id': '69d122f12e',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'}

In [6]:
documents = documents_llm

In [7]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [9]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [10]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [12]:
import json

user_prompt = json.dumps(doc)

In [15]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [16]:
result = response.output_parsed

print(result)

questions=['I just found this course late — can I still join now?', 'If I join after the course started, is there still a way to get a certificate?', 'Do I need to submit my project before submissions close in order to get certified?', 'Can latecomers still take part in the course, or is it too late to join?', 'Is it okay to start the course now if I missed the beginning, and what do I need for the certificate?']


In [17]:
print(result.questions)

['I just found this course late — can I still join now?', 'If I join after the course started, is there still a way to get a certificate?', 'Do I need to submit my project before submissions close in order to get certified?', 'Can latecomers still take part in the course, or is it too late to join?', 'Is it okay to start the course now if I missed the beginning, and what do I need for the certificate?']


In [18]:
from evaluation_utils import llm_structured

In [19]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course late — can I still sign up and follow along?', 'Is it too late to join the class if I missed the start date?', 'Can I still take the course now, and what if I want the certificate too?', 'If I join after the course has already started, can I still get a certificate?', 'What do I need to do in order to be eligible for the course certificate if I’m joining late?']


In [20]:
usage.input_tokens, usage.output_tokens

(207, 102)

In [21]:
from evaluation_utils import calc_price

In [22]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000459, 'total_cost': 0.00061425}

In [23]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course late — can I still sign up and follow along?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to join the class if I missed the start date?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take the course now, and what if I want the certificate too?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do in order to be eligible for the course certificate if I’m joining late?',
  'document': '74eb249bbf'}]

In [24]:
from evaluation_utils import llm_structured_retry

In [25]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [26]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [28]:
len(ground_truth)

25

In [29]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [30]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [32]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [33]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08534850000000001

In [34]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08534850000000001

In [35]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [37]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)